# ReW no House Prices

Regression WiSARD pura, sem Kalman. O objetivo deste notebook é entender o
mecanismo e estabelecer a linha de base contra a qual a versão com filtro de
Kalman será comparada depois.

Decisões fixadas (o porquê de cada uma está em `Literatura`):

| decisão | escolha |
|---|---|
| modelo | ReW apenas, sem CReW e sem ensembles |
| agregação | `SimpleMean`: `Σ soma(y) / Σ contador`, média aritmética sobre os exemplos |
| features | 30 numéricas (6 das 36 descartadas) |
| NaN | mediana **do treino** em `LotFrontage` e `MasVnrArea` |
| split | 70/30, sem estratificação |
| alvo | dólares crus, sem log |
| seeds | 5 nesta fase, 30 quando for para a dissertação |

O número obtido aqui **não** é comparável ao 0.278 da Tabela 5.3 da tese: aqui
só usamos atributos numéricos, a tese não informa como tratou os NaN, e ela quase
certamente transformou o alvo.

O `rew/models.py` é um port fiel de `src/models/regressionwisard/` do wisardpkg,
branch `develop` (a ReW não está no `master`). `ADDRESS_SIZE = 17` é o valor
publicado na Tabela 1 do ESANN 2019 para a melhor ReW.

In [129]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from rew import RegressionWiSARD, SimpleMean, all_metrics
from rew import data

SEEDS = [0, 1, 2, 3, 4]      # subir para range(30) na bateria final
ADDRESS_SIZE = 17            # bits por RAM node (n). 17 é o valor publicado
                             # para a melhor ReW no ESANN 2019 (KDD18)
RESOLUTION = 16              # thermometer size, bits por atributo

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Verificação: as cinco casas do arquivo de literatura

Antes de qualquer métrica, o código tem que reproduzir o exemplo que foi feito
na mão. Cinco casas, uma feature (metragem), thermometer de 6 bits, address size
2, mapping identidade. A predição esperada para uma casa de 1500 ft² é
**$191.250**.

In [130]:
X_toy = np.array([[1200.], [1800.], [3000.], [900.], [3800.]])
y_toy = np.array([150000., 210000., 410000., 95000., 520000.])
mapping_toy = [0, 1, 2, 3, 4, 5]   # identidade: bits 1-2 -> A, 3-4 -> B, 5-6 -> C

toy = RegressionWiSARD(address_size=2, resolution=6, mapping=mapping_toy).fit(X_toy, y_toy)

pred = toy.predict(np.array([[1500.]]))[0]
print(f"predição: ${pred:,.2f}")
assert abs(pred - 191250.0) < 1e-6, "não bateu com o exemplo da literatura"
print("bate com o passo a passo do arquivo Literatura")

predição: $191,250.00
bate com o passo a passo do arquivo Literatura


### O que ficou guardado em cada RAM node

As três tabelas abaixo são a memória inteira do modelo depois do treino. Cada
address guarda **dois** números independentes, `counter` e `sum(y)`. Não existe
nenhum resumo por RAM: o que existe são estas linhas.

In [131]:
from rew.encoder import apply_encoders

Xb_toy = apply_encoders(X_toy, toy._encoders)
addr_toy = toy._addresses(Xb_toy)

print("vetor binário e addresses por casa\n")
for nome, bits, addrs in zip(["H1", "H2", "H3", "H4", "H5"], Xb_toy, addr_toy):
    print(f"  {nome}  {''.join(map(str, bits))}   "
          f"A={addrs[0]:02b}  B={addrs[1]:02b}  C={addrs[2]:02b}")

print()
for r, nome in enumerate("ABC"):
    print(f"RAM {nome}   (só os addresses visitados existem na memória)")
    for a, (c, sy, _) in sorted(toy._memory[r].items()):
        print(f"   address {a:02b}: counter={int(c)}  sum(y)=${sy:,.0f}")
    print()

vetor binário e addresses por casa

  H1  000000   A=00  B=00  C=00
  H2  100000   A=01  B=00  C=00
  H3  111100   A=11  B=11  C=00
  H4  000000   A=00  B=00  C=00
  H5  111110   A=11  B=11  C=01

RAM A   (só os addresses visitados existem na memória)
   address 00: counter=2  sum(y)=$245,000
   address 01: counter=1  sum(y)=$210,000
   address 11: counter=2  sum(y)=$930,000

RAM B   (só os addresses visitados existem na memória)
   address 00: counter=3  sum(y)=$455,000
   address 11: counter=2  sum(y)=$930,000

RAM C   (só os addresses visitados existem na memória)
   address 00: counter=4  sum(y)=$865,000
   address 01: counter=1  sum(y)=$520,000



In [132]:
# a leitura da consulta e a agregação final, passo a passo
q = np.array([[1500.]])
counters, sums = toy._read(apply_encoders(q, toy._encoders))

print("razões locais (etapa 14), uma por RAM node:")
for nome, c, s in zip("ABC", counters[0], sums[0]):
    if c:
        print(f"   RAM {nome}: ${s:,.0f} / {int(c)} = ${s / c:,.2f}")
    else:
        print(f"   RAM {nome}: don't know answer")

print(f"\nagregação (etapa 15): ${sums[0].sum():,.0f} / {int(counters[0].sum())} "
      f"= ${sums[0].sum() / counters[0].sum():,.2f}")
print("\nA SimpleMean soma todos os y e divide pela contagem de todos eles.")
print("Sobre os EXEMPLOS de treino, é média aritmética comum: cada casa entra uma vez.")
print("Sobre as RAM NODES, o efeito é outro: a que viu mais casas influencia mais,")
print("não por um peso explícito, mas por contribuir com mais parcelas na soma.")

razões locais (etapa 14), uma por RAM node:
   RAM A: $210,000 / 1 = $210,000.00
   RAM B: $455,000 / 3 = $151,666.67
   RAM C: $865,000 / 4 = $216,250.00

agregação (etapa 15): $1,530,000 / 8 = $191,250.00

A SimpleMean soma todos os y e divide pela contagem de todos eles.
Sobre os EXEMPLOS de treino, é média aritmética comum: cada casa entra uma vez.
Sobre as RAM NODES, o efeito é outro: a que viu mais casas influencia mais,
não por um peso explícito, mas por contribuir com mais parcelas na soma.


## 2. House Prices: carregamento e pré-processamento

1460 linhas, 81 colunas. Das 38 numéricas, tiramos `Id` e `SalePrice`, sobram 36
features. Descartamos 6 e ficamos com 30.

In [133]:
X, y = data.load_house_prices()

print(f"linhas: {len(X)}   features: {X.shape[1]}")
print(f"alvo: ${y.min():,.0f} a ${y.max():,.0f}   mediana ${y.median():,.0f}")
print(f"\ndescartadas ({len(data.DROP)}): {data.DROP}")
print(f"  código categórico disfarçado de número: {data.DROP_CODIGO_CATEGORICO}")
print(f"  NaN = ausência, não dado faltante:      {data.DROP_NAN_SEMANTICO}")
print(f"  quase constantes (>=96% zeros):         {data.DROP_QUASE_CONSTANTE}")
print(f"\nNaN restantes, imputados pela mediana do treino:")
print(X.isna().sum()[X.isna().sum() > 0].to_string())

linhas: 1460   features: 30
alvo: $34,900 a $755,000   mediana $163,000

descartadas (6): ['MSSubClass', 'GarageYrBlt', 'PoolArea', '3SsnPorch', 'LowQualFinSF', 'MiscVal']
  código categórico disfarçado de número: ['MSSubClass']
  NaN = ausência, não dado faltante:      ['GarageYrBlt']
  quase constantes (>=96% zeros):         ['PoolArea', '3SsnPorch', 'LowQualFinSF', 'MiscVal']

NaN restantes, imputados pela mediana do treino:
LotFrontage    259
MasVnrArea       8


In [134]:
print(f"as {X.shape[1]} features usadas:\n")
resumo = pd.DataFrame({
    "distintos": X.nunique(),
    "min": X.min(),
    "max": X.max(),
    "%zeros": (X == 0).mean().round(3),
})
print(resumo.to_string())

as 30 features usadas:

               distintos     min       max  %zeros
LotFrontage          110    21.0     313.0   0.000
LotArea             1073  1300.0  215245.0   0.000
OverallQual           10     1.0      10.0   0.000
OverallCond            9     1.0       9.0   0.000
YearBuilt            112  1872.0    2010.0   0.000
YearRemodAdd          61  1950.0    2010.0   0.000
MasVnrArea           327     0.0    1600.0   0.590
BsmtFinSF1           637     0.0    5644.0   0.320
BsmtFinSF2           144     0.0    1474.0   0.886
BsmtUnfSF            780     0.0    2336.0   0.081
TotalBsmtSF          721     0.0    6110.0   0.025
1stFlrSF             753   334.0    4692.0   0.000
2ndFlrSF             417     0.0    2065.0   0.568
GrLivArea            861   334.0    5642.0   0.000
BsmtFullBath           4     0.0       3.0   0.586
BsmtHalfBath           3     0.0       2.0   0.944
FullBath               4     0.0       3.0   0.006
HalfBath               3     0.0       2.0   0.625
Bedroom

Repare nas colunas de cardinalidade baixa, como `FullBath` (4 valores distintos)
ou `BsmtHalfBath` (3). Com thermometer de 16 bits e faixa de 0 a 3, o encoder
produz só 4 padrões distintos gastando 16 bits de retina. Não está errado, é
desperdício. É exatamente o que thermometer por feature resolveria, e é o que a
tese registra que não teve tempo de testar.

## 3. Uma seed, para olhar o modelo de perto

In [135]:
X_tr, X_te, y_tr, y_te = data.split(X, y, seed=0)
X_tr, X_te = data.impute_train_median(X_tr, X_te)

modelo = RegressionWiSARD(
    address_size=ADDRESS_SIZE, resolution=RESOLUTION, seed=0, mean=SimpleMean()
).fit(X_tr.values, y_tr.values)

print(f"treino: {len(X_tr)}   teste: {len(X_te)}")
print(f"retina size: {modelo.retina_size_} bits "
      f"({X.shape[1]} features x {RESOLUTION} bits)")
usados = modelo.addresses_used()
print(f"RAM nodes (N): {modelo.n_rams_}   "
      f"addresses possíveis por RAM: {2 ** ADDRESS_SIZE:,}")
print(f"addresses realmente usados por RAM: média {usados.mean():.1f} "
      f"(de {len(X_tr)} casas de treino)")
print(f"memória: {modelo.memory_bytes() / 1024:.1f} KB")
print("\nA memória é esparsa (dicionário), como o std::map do wisardpkg.")
print("Densa seria inviável: 2^17 posições por RAM node.")

treino: 1022   teste: 438
retina size: 480 bits (30 features x 16 bits)
RAM nodes (N): 29   addresses possíveis por RAM: 131,072
addresses realmente usados por RAM: média 130.8 (de 1022 casas de treino)
memória: 118.5 KB

A memória é esparsa (dicionário), como o std::map do wisardpkg.
Densa seria inviável: 2^17 posições por RAM node.


In [136]:
y_pred = modelo.predict(X_te.values)

for k, v in all_metrics(y_te.values, y_pred).items():
    if k in ("R2",):
        print(f"  {k:14s} {v:.4f}")
    elif k.startswith("dentro"):
        print(f"  {k:14s} {v:.1%}")
    else:
        print(f"  {k:14s} ${v:,.0f}")

  MAE            $41,032
  RMSE           $62,922
  R2             0.4168
  dentro_de_10%  28.8%


### Diagnóstico de esparsidade

Fração das `N` RAM nodes que caíram em address nunca visto. Não é métrica de
erro: é a medida direta do fenômeno que o ESANN 2019 e a tese mencionam sem
formalizar, de que a confiança cai quando a rede fica esparsa. É a linha de base
contra a qual a incerteza `p` do Kalman vai ser comparada.

In [137]:
dk = modelo.dont_know_fraction(X_te.values)

print(f"don't know por amostra: média {dk.mean():.1%}, "
      f"mediana {np.median(dk):.1%}, máximo {dk.max():.1%}")
print(f"amostras com predição = 0 (todas as RAMs falharam): {(y_pred == 0).sum()}")

pior = np.argsort(np.abs(y_te.values - y_pred))[-5:][::-1]
print("\nas 5 piores predições:")
for i in pior:
    print(f"   real ${y_te.values[i]:>9,.0f}   previsto ${y_pred[i]:>9,.0f}   "
          f"don't know {dk[i]:.0%}")

don't know por amostra: média 6.1%, mediana 3.4%, máximo 75.9%
amostras com predição = 0 (todas as RAMs falharam): 0

as 5 piores predições:
   real $  745,000   previsto $  268,904   don't know 48%
   real $  555,000   previsto $  212,705   don't know 34%
   real $  611,657   previsto $  271,741   don't know 24%
   real $  538,000   previsto $  283,430   don't know 38%
   real $  465,000   previsto $  218,535   don't know 7%


## 4. As cinco seeds

In [138]:
linhas = []
for seed in SEEDS:
    Xtr, Xte, ytr, yte = data.split(X, y, seed=seed)
    Xtr, Xte = data.impute_train_median(Xtr, Xte)

    m = RegressionWiSARD(
        address_size=ADDRESS_SIZE, resolution=RESOLUTION, seed=seed, mean=SimpleMean()
    ).fit(Xtr.values, ytr.values)

    yp = m.predict(Xte.values)
    r = all_metrics(yte.values, yp)
    r["seed"] = seed
    r["dont_know"] = m.dont_know_fraction(Xte.values).mean()
    linhas.append(r)

res = pd.DataFrame(linhas).set_index("seed")
print(res.to_string())
print()
print("média e desvio sobre as seeds:")
print(res.agg(["mean", "std"]).to_string())

               MAE          RMSE        R2  dentro_de_10%  dont_know
seed                                                                
0     41032.411074  62922.423518  0.416816       0.287671   0.060542
1     39495.766403  60946.103928  0.479794       0.315068   0.053929
2     37178.114582  57512.499516  0.489062       0.353881   0.052984
3     32308.412006  47548.142903  0.577757       0.365297   0.061014
4     36780.708350  52263.037848  0.488705       0.312785   0.044796

média e desvio sobre as seeds:
               MAE          RMSE        R2  dentro_de_10%  dont_know
mean  37359.082483  56238.441543  0.490427       0.326941   0.054653
std    3314.185398   6320.325325  0.057365       0.031939   0.006626


### Comparação com o preditor constante

O R² já responde isso, mas vale ver em dólares: qual seria o MAE de um modelo
que ignora todas as features e chuta sempre a mediana dos preços de treino?
Qualquer coisa que a ReW faça precisa ser melhor do que isso para ter valor.

In [139]:
from rew.metrics import mae

for seed in SEEDS[:1]:
    Xtr, Xte, ytr, yte = data.split(X, y, seed=seed)
    baseline = np.full(len(yte), ytr.median())
    print(f"chute da mediana do treino: MAE ${mae(yte.values, baseline):,.0f}")
    print(f"ReW (seed {seed}):            MAE ${res.loc[seed, 'MAE']:,.0f}")

chute da mediana do treino: MAE $56,469
ReW (seed 0):            MAE $41,032


## Nota

O modelo tem três coisas implementadas que não são usadas aqui e vêm desligadas
por padrão, todas presentes na referência:

- `min_zero` / `min_one`: exigem um mínimo de zeros ou uns na formação do
  address para a RAM node contribuir;
- `steps`: passadas de correção de resíduo depois do treino, a única atualização
  iterativa que existe na ReW;
- as outras sete médias em `rew/means.py`, incluindo a `Median`, que foi a que
  deu o melhor resultado da ReW no House Prices segundo a Tabela 5.3 da tese.

## 5. Kalman-ReW

### Por que a agregação da ReW não transfere direto

Na ReW a célula guarda `S_k`, a **soma** dos `y` daquele endereço. Soma é
extensiva: cresce junto com o contador. Por isso

```
        Σ_k S_k
ŷ  =  ───────────
        Σ_k c_k
```

é uma média agrupada legítima, o total de `y` sobre o total de exemplos.

Com Kalman, a célula não guarda soma nenhuma. Ela guarda um **estado** `y_k`,
que já é uma média e não cresce com o número de visitas. Somar `y_k` entre RAM
nodes e dividir por `Σc_k` seria dimensionalmente errado: o numerador não
acompanha o denominador.

A forma correta é uma média ponderada explícita, com os pesos normalizados:

```
        Σ_k  c_k · y_k
ŷ  =  ──────────────────
           Σ_k c_k
```

E aqui está o ponto: para isso funcionar, `c_k` também precisa ser um **estado
convergido**, limitado a [0,1], e não uma contagem sem teto. Um endereço visto
1069 vezes não pode valer 1069 votos.

### Os dois estados da célula

Cada célula guarda quatro números, e os dois primeiros são estados do filtro:

| campo | observação `z` | converge para | papel |
|---|---|---|---|
| `c_k` | `1.0` | 1 | peso da RAM node na agregação |
| `y_k` | `y_i` | média local de `y` | estimativa da RAM node |
| `p_c`, `p_y` | — | — | incertezas de cada filtro |

O `c_k` é o mesmo mecanismo do KalWeNN de classificação: observação fixa em
`1.0`, estado saturando. O `y_k` é a novidade da regressão: observação é o valor
contínuo.

### Filtrar só `y` reproduz a ReW quando `q = 0`

Com ruído de processo zero, o ganho vira `k = 1/(n+1)`, que é a fórmula
incremental da média aritmética. Ou seja, o filtro calcula exatamente
`soma(y)/contador`, que é o que a ReW já faz.

A célula abaixo mostra os dois casos: `q = 0`, que reproduz, e `q = 0.1`, que
não reproduz. A diferença do segundo não vem do filtro, vem do esquecimento que
`q > 0` introduz.

In [140]:
from rew.kalman import KalmanReW, KalmanReW_apenasY

X_tr, X_te, y_tr, y_te = data.split(X, y, seed=0)
X_tr, X_te = data.impute_train_median(X_tr, X_te)

rew_ = RegressionWiSARD(address_size=ADDRESS_SIZE, resolution=RESOLUTION,
                        seed=0, mean=SimpleMean()).fit(X_tr.values, y_tr.values)
base = rew_.predict(X_te.values)

for q in [0.0, 0.1]:
    so_y = KalmanReW_apenasY(address_size=ADDRESS_SIZE, resolution=RESOLUTION,
                             seed=0, mean=SimpleMean(), q=q, r=1.0).fit(X_tr.values, y_tr.values)
    d = np.abs(base - so_y.predict(X_te.values)).max()
    print(f"q={q:<4} diferença para a ReW: {d:>12.3e}   "
          f"{'idêntico' if d < 1e-6 else 'diferente'}")

assert np.abs(base - KalmanReW_apenasY(
    address_size=ADDRESS_SIZE, resolution=RESOLUTION, seed=0,
    mean=SimpleMean(), q=0.0, r=1.0).fit(X_tr.values, y_tr.values).predict(X_te.values)
).max() < 1e-6, "com q=0 deveria reproduzir a ReW"

print("\nCom q=0 o filtro é a média corrente, ou seja, a própria ReW.")
print("A diferença que aparece com q>0 vem do ESQUECIMENTO, não do filtro em si:")
print("a célula passa a pesar mais os exemplos que chegaram por último.")

ImportError: cannot import name 'KalmanReW_apenasY' from 'rew.kalman' (/Users/helenaburock/Git/kwisard/KalWeNN/RegressionKalWeNN/rew/kalman.py)

### O efeito vem do peso normalizado

`r_c` controla a velocidade de saturação do peso. Grande satura devagar e se
aproxima do contador cru; pequeno satura rápido e se aproxima do peso uniforme.

In [ ]:
from rew.means import UnweightedMean
from rew.metrics import mae as _mae, rmse as _rmse, r2 as _r2

def avalia(construtor):
    acc = []
    for seed in SEEDS:
        Xtr, Xte, ytr, yte = data.split(X, y, seed=seed)
        Xtr, Xte = data.impute_train_median(Xtr, Xte)
        m = construtor(seed).fit(Xtr.values, ytr.values)
        yp, yt = m.predict(Xte.values), yte.values
        acc.append((_mae(yt, yp), _rmse(yt, yp), _r2(yt, yp), yp.std() / yt.std()))
    a = np.array(acc)
    return {"MAE": a[:, 0].mean(), "MAE_dp": a[:, 0].std(), "RMSE": a[:, 1].mean(),
            "R2": a[:, 2].mean(), "dp_pred/dp_real": a[:, 3].mean()}

linhas = [dict(modelo="ReW (peso = contador cru)", **avalia(
    lambda s: RegressionWiSARD(address_size=ADDRESS_SIZE, resolution=RESOLUTION,
                               seed=s, mean=SimpleMean())))]

for rc in [8.0, 2.0, 0.5, 0.05]:
    linhas.append(dict(modelo=f"Kalman em y e em c  (r_c={rc})", **avalia(
        lambda s, rc=rc: KalmanReW(address_size=ADDRESS_SIZE, resolution=RESOLUTION,
                                       seed=s, q_c=1e-3, r_c=rc, q_y=0.0, r_y=1.0))))

linhas.append(dict(modelo="ReW (peso uniforme, o limite)", **avalia(
    lambda s: RegressionWiSARD(address_size=ADDRESS_SIZE, resolution=RESOLUTION,
                               seed=s, mean=UnweightedMean()))))

print(pd.DataFrame(linhas).to_string(
    index=False, float_format=lambda v: f"{v:,.4f}" if abs(v) < 10 else f"{v:,.0f}"))

                        modelo    MAE  MAE_dp   RMSE     R2  dp_pred/dp_real
     ReW (peso = contador cru) 37,359   2,964 56,238 0.4904           0.3697
 Kalman em y e em c  (r_c=8.0) 29,994   2,038 46,047 0.6588           0.5070
 Kalman em y e em c  (r_c=2.0) 28,650   1,945 43,687 0.6928           0.5410
 Kalman em y e em c  (r_c=0.5) 27,980   1,896 42,407 0.7105           0.5612
Kalman em y e em c  (r_c=0.05) 27,681   1,857 41,811 0.7184           0.5717
 ReW (peso uniforme, o limite) 27,644   1,852 41,736 0.7194           0.5732


### Como ler

**A troca do peso é o que produz o efeito.** Sair do contador cru para o peso
saturado derruba o MAE em cerca de 26% e sobe o R² de 0,49 para 0,72. O desvio
entre seeds cai junto, então o modelo também fica mais estável.

**A causa está medida.** Com peso cru, existem endereços com 1069 dos 1168
exemplos de treino. A estimativa local desses blocos é praticamente a média
global, e eles dominam a soma. As RAM nodes específicas, que são as que sabem
distinguir uma casa cara, não têm peso para se fazer ouvir.

**Mas o ganho é monotônico em `r_c`.** Quanto mais satura, melhor, e a última
linha é o limite: peso 1 para todo mundo. O Kalman em `c` chega perto desse
limite por baixo, sem ultrapassar. Com `r_c = 0.05` a diferença para o uniforme
é de algumas dezenas de dólares num MAE de 27 mil, dentro do ruído entre seeds.

Ou seja, o mecanismo que funciona é **limitar o peso**, e a forma mais simples
possível de limitar (fixar tudo em 1) entrega o mesmo que o filtro.